#AI DATA CLEANER:



Build an intelligent data cleaning agent that detects problems in datasets and fixes them automatically using ai prompt


AI-Powered Intelligent Data Cleaning Agent using Pandas and Groq LLM

In [1]:
!pip install groq pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import json
from groq import Groq

In [3]:
GROQ_API_KEY = "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"

client = Groq(
    api_key=GROQ_API_KEY
)

In [4]:
df=pd.read_csv('messy_sales_data.csv')
df.head()

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [5]:
print("Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

Shape:
(30, 9)

Columns:
Index(['order_id', 'customer_name', 'product', 'category', 'quantity',
       'unit_price', 'order_date', 'city', 'sales_rep'],
      dtype='object')

Data Types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object


In [6]:
#create dataset profile
profile = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_values":
        df.isnull().sum().to_dict(),
    "duplicates":
        int(df.duplicated().sum()),
    "data_types":
        df.dtypes.astype(str).to_dict()
}

profile

{'rows': 30,
 'columns': 9,
 'missing_values': {'order_id': 0,
  'customer_name': 2,
  'product': 1,
  'category': 1,
  'quantity': 3,
  'unit_price': 0,
  'order_date': 0,
  'city': 0,
  'sales_rep': 0},
 'duplicates': 0,
 'data_types': {'order_id': 'int64',
  'customer_name': 'object',
  'product': 'object',
  'category': 'object',
  'quantity': 'float64',
  'unit_price': 'int64',
  'order_date': 'object',
  'city': 'object',
  'sales_rep': 'object'}}

In [8]:
#send profile to groq
prompt = f"""
You are an AI Data Cleaning Agent.

Analyze the dataset profile.

{profile}

Return JSON only.

Format:

{{
 "steps":[
  {{
   "action":"remove_duplicates"
  }},
  {{
   "column":"column_name",
   "action":"fill_mode"
  }}
 ]
}}
"""

In [9]:
#Ask Groq for Cleaning Instructions

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

result = response.choices[0].message.content

print(result)

```json
{
 "steps":[
  {
   "column":"customer_name",
   "action":"fill_mode"
  },
  {
   "column":"product",
   "action":"fill_mode"
  },
  {
   "column":"category",
   "action":"fill_mode"
  },
  {
   "column":"quantity",
   "action":"fill_mean"
  }
 ]
}
```


In [10]:
#Convert AI Response to JSON
import re

json_text = re.search(
    r'\{.*\}',
    result,
    re.DOTALL
).group()

instructions = json.loads(json_text)

instructions

{'steps': [{'column': 'customer_name', 'action': 'fill_mode'},
  {'column': 'product', 'action': 'fill_mode'},
  {'column': 'category', 'action': 'fill_mode'},
  {'column': 'quantity', 'action': 'fill_mean'}]}

In [11]:
#Execute AI Cleaning Instructions
for step in instructions["steps"]:

    action = step["action"]

    if action == "remove_duplicates":
        df = df.drop_duplicates()

    elif action == "fill_mode":

        col = step["column"]

        if col in df.columns:

            value = df[col].mode()[0]

            df[col] = df[col].fillna(value)

    elif action == "fill_median":

        col = step["column"]

        if col in df.columns:

            value = df[col].median()

            df[col] = df[col].fillna(value)

print("AI Cleaning Completed")

AI Cleaning Completed


In [12]:
#Standardize Text Columns
for col in df.select_dtypes(include="object"):

    df[col] = df[col].astype(str)

    df[col] = df[col].str.strip()

    df[col] = df[col].str.title()

print("Text Standardized")

Text Standardized


In [13]:
#Calculate Data Quality Score
total_cells = (
    df.shape[0] *
    df.shape[1]
)

missing_cells = (
    df.isnull().sum().sum()
)

quality_score = (
    (total_cells - missing_cells)
    / total_cells
) * 100

print(
    f"Quality Score = {quality_score:.2f}%"
)

Quality Score = 98.89%


In [14]:
#Generate AI Cleaning Report
report_prompt = f"""
Generate a professional report.

Rows: {df.shape[0]}
Columns: {df.shape[1]}
Quality Score: {quality_score:.2f}

Mention:
1. Missing values fixed
2. Duplicates removed
3. Text standardized
4. Dataset ready for analysis
"""

In [15]:
#Get AI Report
report = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role":"user",
            "content":report_prompt
        }
    ]
)

print(
    report.choices[0]
    .message.content
)

**Data Quality Report**

**Introduction:**
This report outlines the results of the data preprocessing phase, which aimed to enhance the quality and reliability of the dataset. The dataset consists of 30 rows and 9 columns, providing a comprehensive foundation for further analysis.

**Data Quality Enhancements:**
Several key enhancements have been made to the dataset to ensure its accuracy and consistency:

1. **Missing Values Resolution:** All missing values within the dataset have been identified and addressed. This has improved the overall completeness of the data, reducing the risk of biased or inaccurate analysis.
2. **Duplicate Removal:** A thorough review of the dataset was conducted to identify and remove duplicate entries. This process has ensured that each data point is unique, enhancing the precision of future analyses.
3. **Text Standardization:** Text data within the dataset has been standardized to ensure consistency in formatting and style. This standardization facilitate

In [16]:
#Save Cleaned Dataset
df.to_csv(
    "cleaned_sales_data.csv",
    index=False
)

print(
    "Dataset Saved Successfully"
)

Dataset Saved Successfully
